# V19: matched readout diagnostic

Compare unadapted Qwen3-4B with our v18 external-only and final adapters. Fixed three-layer standardized logistic readouts, held-out v18 prompts only. No submission. Qwen, LoRA and SupCon retain their original credits; external pool credited to Nishant Dahal and original contributors (CC BY-SA 4.0).

In [ ]:

import os, sys, json, glob, time, subprocess, hashlib, importlib.metadata
from pathlib import Path
from packaging.version import Version
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[name] = "2"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
try: torchao_version = importlib.metadata.version("torchao")
except importlib.metadata.PackageNotFoundError: torchao_version = None
if torchao_version is not None and Version(torchao_version) < Version("0.16.0"):
    subprocess.run([sys.executable,"-m","pip","uninstall","-y","torchao"],check=True)
import torch, peft, transformers
assert torch.cuda.device_count()==2
print({"torch":torch.__version__,"peft":peft.__version__,"transformers":transformers.__version__},flush=True)
plan = json.loads("{\"experiment\": \"improvement-v19\", \"kind\": \"representation/readout diagnostic after v18\", \"model\": \"qwen-lm/qwen-3/Transformers/4b-base/1\", \"upstream_kernel\": \"adithyagiri/name-that-llm-contrastive-v18\", \"upstream_version\": 2, \"variants\": [\"base\", \"ce_external\", \"ce_final\", \"supcon_external\", \"supcon_final\"], \"layers\": [12, 24, 36], \"max_length\": 256, \"batch_size\": 8, \"C\": 0.02, \"max_iter\": 1000, \"pilot_fold\": 0, \"seed\": 1901, \"primary_readout\": \"Equal mean of three layer log probabilities; per-layer results diagnostic only\", \"preprocessing\": \"StandardScaler fit on 2159 host fitting rows only; same fit/validation rows for all five backbones\", \"promotion\": \"An adapted primary readout must beat the identical base readout by >=0.01 and reach >=0.4044444444 before broader confirmation. No pilot submission.\", \"limitations\": [\"One repeatedly inspected fold\", \"Final host adapters are valid only on v18's held-out fold; their training rows must never be used as OOF evidence\", \"External-only adapters can support later folds, but v18's global host-text overlap exclusion is retained\", \"This tests readout quality, not new backbone training\"], \"sources\": [\"https://arxiv.org/html/2606.10794v1\", \"https://arxiv.org/html/2004.11362v5\", \"https://github.com/HobbitLong/SupContrast\", \"https://github.com/microsoft/LoRA\"], \"adapter_sha256\": {\"ce_external\": \"ea201bafa6acfa4c4dc6c88f7b030322562bf43de9d89e81ace6ab1fefd36f96\", \"ce_final\": \"c1f98a3b6dc5d59e3c7f27d59bc455bae0c438edf0d8897a6aa3e547e19f7c83\", \"supcon_external\": \"353f3e3bacab0660d60b8b8e4a90e4d258f26fa5047a4010db3f2227b7bfb6d5\", \"supcon_final\": \"2aae37e987d7cbfc55f6d83afcbdb8e1d7545aa1a22ebc173e167683399047fc\"}, \"upstream_worker_sha256\": \"a79591118969d7e2b687523a64c46f68399a217c36972d37e300aaba7d440b34\", \"host_sha256\": \"40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707\"}")
matches = [p for p in glob.glob("/kaggle/input/**/plan.json",recursive=True)
           if json.loads(Path(p).read_text()).get("experiment")=="improvement-v18"]
assert len(matches)==1,matches
upstream = Path(matches[0]).parent
assert hashlib.sha256((upstream/"worker.py").read_bytes()).hexdigest()==plan["upstream_worker_sha256"]
plan["upstream_path"] = str(upstream)
paths = [p for p in glob.glob("/kaggle/input/**/config.json",recursive=True) if "/4b-base/" in p.lower()]
assert len(paths)==1,paths
plan["model_dir"] = str(Path(paths[0]).parent)
host = next(p for p in glob.glob("/kaggle/input/**/train.csv",recursive=True) if "name-that-llm" in p)
assert hashlib.sha256(Path(host).read_bytes()).hexdigest()==plan["host_sha256"]
plan["host_path"] = host
Path("plan.json").write_text(json.dumps(plan,indent=2))
Path("worker.py").write_text('"""Matched frozen readouts for base and v18 adapters; one fixed host fold."""\nimport gc\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport time\nimport warnings\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.exceptions import ConvergenceWarning\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.preprocessing import StandardScaler\nfrom transformers import AutoModel, AutoTokenizer\nfrom peft import PeftModel\n\n\ndef main():\n    plan = json.loads(Path("plan.json").read_text())\n    host = pd.read_csv(plan["host_path"], dtype={"id":str})\n    upstream = Path(plan["upstream_path"])\n    original_plan = json.loads((upstream / "plan.json").read_text())\n    assert original_plan["pilot_fold"] == plan["pilot_fold"]\n    fitting = pd.read_parquet(upstream / "training_frame.parquet")\n    validation = pd.read_parquet(upstream / "validation_frame.parquet")\n    fit_ids = set(fitting.loc[fitting.stage == "host", "id"])\n    val_ids = set(validation.id)\n    assert fit_ids.isdisjoint(val_ids) and fit_ids | val_ids == set(host.id)\n    assert len(fit_ids) == 2159 and len(val_ids) == 540\n    train_mask = host.id.isin(fit_ids).to_numpy()\n    valid_mask = host.id.isin(val_ids).to_numpy()\n    assert host.loc[valid_mask,"id"].tolist() == validation.id.tolist()\n    labels = np.asarray(original_plan["labels"])\n    assert np.array_equal(labels, np.sort(host.model.unique()))\n    tok = AutoTokenizer.from_pretrained(plan["model_dir"], local_files_only=True)\n    tok.padding_side = "right"\n    if tok.pad_token_id is None: tok.pad_token = tok.eos_token\n    tokenized = tok(host.text.tolist(), truncation=True, max_length=plan["max_length"]-1,\n                    add_special_tokens=False)["input_ids"]\n    tokenized = [ids+[tok.eos_token_id] for ids in tokenized]\n    order = np.argsort([len(ids) for ids in tokenized], kind="stable")\n    checkpoint_paths = {"ce_external":"ce/external_only/adapter", "ce_final":"ce/final/adapter",\n                        "supcon_external":"ce_supcon/external_only/adapter", "supcon_final":"ce_supcon/final/adapter"}\n    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False\n    assert torch.cuda.device_count() == 1\n    for variant in json.loads(os.environ["V19_VARIANTS"]):\n        started = time.monotonic()\n        print(json.dumps({"variant":variant,"event":"loading"}), flush=True)\n        model = AutoModel.from_pretrained(plan["model_dir"], torch_dtype=torch.float16,\n                                         attn_implementation="sdpa", local_files_only=True)\n        if variant != "base":\n            checkpoint = upstream / checkpoint_paths[variant]\n            weight = checkpoint / "adapter_model.safetensors"\n            assert hashlib.sha256(weight.read_bytes()).hexdigest() == plan["adapter_sha256"][variant]\n            model = PeftModel.from_pretrained(model, checkpoint, is_trainable=False)\n        model = model.cuda().eval()\n        width = model.config.hidden_size\n        features = np.empty((len(host),len(plan["layers"]),width), dtype=np.float32)\n        with torch.inference_mode():\n            for start in range(0,len(host),plan["batch_size"]):\n                ids = order[start:start+plan["batch_size"]]\n                enc = {k:v.cuda() for k,v in tok.pad({"input_ids":[tokenized[i] for i in ids]},\n                                                    padding=True,return_tensors="pt").items()}\n                with torch.autocast("cuda", dtype=torch.float16):\n                    states = model(**enc, output_hidden_states=True, use_cache=False).hidden_states\n                mask = enc["attention_mask"].clone(); mask[:,0] = 0\n                for j,layer in enumerate(plan["layers"]):\n                    h = states[layer].float()\n                    features[ids,j] = ((h*mask.unsqueeze(-1)).sum(1)/mask.sum(1,keepdim=True).clamp_min(1)).cpu().numpy()\n                del states\n                if start % 400 == 0:\n                    print(json.dumps({"variant":variant,"completed":min(start+len(ids),len(host)),\n                                      "total":len(host),"seconds":time.monotonic()-started}), flush=True)\n        assert np.isfinite(features).all()\n        np.save(variant+"_features.npy", features)\n        del model\n        gc.collect(); torch.cuda.empty_cache()\n        predictions = []\n        diagnostics = {}\n        for j,layer in enumerate(plan["layers"]):\n            scaler = StandardScaler()\n            X = scaler.fit_transform(features[train_mask,j])\n            V = scaler.transform(features[valid_mask,j])\n            clf = LogisticRegression(C=plan["C"], max_iter=plan["max_iter"], random_state=plan["seed"])\n            with warnings.catch_warnings():\n                warnings.simplefilter("error", ConvergenceWarning)\n                clf.fit(X,host.loc[train_mask,"model"])\n            assert np.array_equal(clf.classes_, labels)\n            scores = clf.predict_log_proba(V)\n            predictions.append(scores)\n            diagnostics[str(layer)] = float((labels[scores.argmax(1)]==host.loc[valid_mask,"model"]).mean())\n        scores = np.mean(predictions,axis=0)\n        accuracy = float((labels[scores.argmax(1)]==host.loc[valid_mask,"model"]).mean())\n        np.savez_compressed(variant+"_validation.npz", ids=host.loc[valid_mask,"id"].to_numpy(dtype=str),\n                            labels=labels.astype(str), scores=scores, per_layer_scores=np.stack(predictions))\n        result = {"variant":variant,"accuracy":accuracy,"per_layer_accuracy":diagnostics,\n                  "seconds":time.monotonic()-started,"fit_rows":int(train_mask.sum()),\n                  "validation_rows":int(valid_mask.sum()),"feature_shape":list(features.shape),\n                  "feature_sha256":hashlib.sha256(Path(variant+"_features.npy").read_bytes()).hexdigest()}\n        Path(variant+"_result.json").write_text(json.dumps(result,indent=2))\n        print(json.dumps(result),flush=True)\n        del features\n        gc.collect()\n\n\nif __name__ == "__main__":\n    main()\n')
assignments = [["base","ce_external","ce_final"],["supcon_external","supcon_final"]]
processes = []
for gpu,variants in enumerate(assignments):
    env = dict(os.environ,CUDA_VISIBLE_DEVICES=str(gpu),V19_VARIANTS=json.dumps(variants))
    processes.append(subprocess.Popen([sys.executable,"-u","worker.py"],env=env))
try:
    while any(p.poll() is None for p in processes):
        if any(p.poll() not in (None,0) for p in processes):
            raise RuntimeError("Readout worker failed; inspect worker traceback")
        time.sleep(10)
    assert all(p.returncode==0 for p in processes)
except BaseException:
    for p in processes:
        if p.poll() is None:p.terminate()
    for p in processes:p.wait()
    raise
results = {v:json.loads(Path(v+"_result.json").read_text()) for v in plan["variants"]}
base = results["base"]["accuracy"]
promoted = [v for v,r in results.items() if v!="base" and r["accuracy"]-base>=0.01 and r["accuracy"]>=0.4044444444]
summary = {"results":results,"promoted_for_confirmation":promoted,"submission_authorized":False}
Path("v19_summary.json").write_text(json.dumps(summary,indent=2))
print(json.dumps(summary),flush=True)
